# What a REX export can show

Prepared for the Data Donation Symposium 2026.

This notebook reads one REX local-download export and walks through four questions a researcher might ask of it. Two are about browsing history, one is about LLM conversations, and one joins the two.

REX collects data in the participant's own browser. History arrives at the privacy level the study's lists allow, and ChatGPT conversations arrive as ChatGPT's own complete message tree. Several things below cannot be seen in the data download ChatGPT itself provides; those are marked.

**To run it:** in Google Colab, run all cells and choose your export file when asked. Colab uploads the file to Google's servers, so use it only for data you are free to share there. On your own computer, set `EXPORT_PATH` below to your export file, then run all cells.

In [ ]:
from pathlib import Path

EXPORT_PATH = Path("rex-export.json")
REPO_RAW_URL = "https://raw.githubusercontent.com/bric-digital/rex-data-donation-notebook/main"

In [ ]:
import sys

if "google.colab" in sys.modules:
    import urllib.request
    from google.colab import files

    urllib.request.urlretrieve(f"{REPO_RAW_URL}/rex_export.py", "rex_export.py")
    uploaded = files.upload()
    EXPORT_PATH = Path(next(iter(uploaded)))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from rex_export import load_export, prompt_reply_pairs

PINK, TEAL, GREEN, MINT, BROWN = "#b42b6f", "#106e7d", "#1ba97f", "#afecda", "#5b2d07"
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "font.size": 10,
})

export = load_export(EXPORT_PATH)
visits, conversations, messages = export.visits, export.conversations, export.messages
local_tz = export.capture_timezone or "UTC"


def span(times):
    return f"{times.min():%Y-%m-%d} to {times.max():%Y-%m-%d}" if len(times) else "none"


pd.DataFrame({
    "records": [len(visits), len(conversations), len(messages)],
    "covers": [span(visits["visit_time"]), span(conversations["create_time"]), span(messages["create_time"])],
}, index=["history visits", "conversations", "messages"])

## 1. What does a donor actually give?

Each history visit reaches the researcher at one of three levels, decided by the study's allow and block lists:

- **Full URL**: the page address and title, for sites on the allow list.
- **Domain only**: the site but not the page, the default.
- **Category only**: a label such as `email` in place of the site, for sites on the block list.

The chart shows the share of each level by week. The privacy design is visible in the data itself, and a researcher can check it against the study protocol.

In [ ]:
TIER_ORDER = ["full", "domain", "category"]
TIER_LABELS = {"full": "Full URL", "domain": "Domain only", "category": "Category only"}
TIER_COLORS = {"full": PINK, "domain": TEAL, "category": MINT}

weekly = (visits
          .assign(week=visits["visit_time"].dt.tz_convert(local_tz).dt.to_period("W").dt.start_time)
          .groupby(["week", "privacy_tier"]).size().unstack(fill_value=0)
          .reindex(columns=TIER_ORDER, fill_value=0))
weekly_share = weekly.div(weekly.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(9, 3.6))
bottom = np.zeros(len(weekly_share))
for tier in TIER_ORDER:
    ax.bar(weekly_share.index, weekly_share[tier], bottom=bottom, width=5,
           color=TIER_COLORS[tier], label=TIER_LABELS[tier])
    bottom += weekly_share[tier].to_numpy()
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.set_title("Share of visits by privacy level, per week")
ax.legend(frameon=False, ncols=3, loc="upper left", bbox_to_anchor=(0, -0.12))
plt.show()

totals = visits["privacy_tier"].value_counts().reindex(TIER_ORDER, fill_value=0)
pd.DataFrame({"visits": totals, "share": (totals / totals.sum()).map("{:.0%}".format)}).rename(index=TIER_LABELS)

Category-only visits by label. These are the sites the study chose not to see, counted without being named.

In [ ]:
visits.loc[visits["privacy_tier"] == "category", "category"].value_counts().rename("visits").to_frame()

## 2. When do people browse and chat, and what does a conversation look like?

The heatmaps show browsing and chat prompts by hour and weekday, each as a share of its own total so the two can be compared even though history covers more days than conversations.

**Timezone caveat.** Times are converted using the timezone recorded when the data was captured. If the participant travelled during the history window, visits from before the trip are shifted by the difference.

In [ ]:
WEEKDAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]


def hour_by_weekday(times):
    local = times.dt.tz_convert(local_tz)
    grid = pd.crosstab(local.dt.dayofweek, local.dt.hour).reindex(index=range(7), columns=range(24), fill_value=0)
    return grid / grid.to_numpy().sum()


prompts = messages[(messages["role"] == "user") & ~messages["is_hidden_step"]]
grids = {"Browsing": hour_by_weekday(visits["visit_time"]), "Chat prompts": hour_by_weekday(prompts["create_time"])}

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for ax, (title, grid) in zip(axes, grids.items()):
    image = ax.imshow(grid, aspect="auto", cmap="RdPu")
    ax.set_title(title)
    ax.set_yticks(range(7), WEEKDAYS)
    ax.set_xticks(range(0, 24, 3), [f"{h:02d}" for h in range(0, 24, 3)])
    ax.set_xlabel(f"hour ({local_tz})")
    fig.colorbar(image, ax=ax, format=plt.FuncFormatter(lambda v, _: f"{v:.0%}"), shrink=0.85)
plt.show()

In [ ]:
visible = messages[~messages["is_hidden_step"] & messages["role"].isin(["user", "assistant"])]
prompts_per_conversation = visible[visible["role"] == "user"].groupby("conversation_id").size()
pairs = prompt_reply_pairs(messages)

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))
CAP = 6
capped = prompts_per_conversation.clip(upper=CAP).value_counts().reindex(range(1, CAP + 1), fill_value=0)
labels = [str(n) for n in range(1, CAP)] + [f"{CAP}+"]
left.bar(labels, capped.values, color=TEAL)
left.set_title("Prompts per conversation")
left.set_xlabel("prompts")
left.set_ylabel("conversations")

right.scatter(pairs["prompt_chars"], pairs["reply_chars"], s=12, alpha=0.5, color=PINK, edgecolors="none")
right.set_xscale("log")
right.set_yscale("log")
right.set_title("Prompt length against reply length")
right.set_xlabel("prompt characters")
right.set_ylabel("reply characters")
plt.show()

ratio = (pairs["reply_chars"] / pairs["prompt_chars"]).median()
print(f"{prompts_per_conversation.eq(1).mean():.0%} of conversations end after one prompt.")
print(f"The median reply is {ratio:.0f} times longer than the prompt it answers.")

## 3. What does the model do behind the scenes?

These fields come from ChatGPT's own message tree, which REX keeps whole.

- **Requested model and the model that answered.** Conversations usually request `auto`; the answering model is recorded on each reply. ChatGPT's own data download has this too.
- **Reasoning time**, where the model reports it. The data download has only a rounded label such as "Worked for 14s"; REX also has exact start and end times.
- **Hidden steps**: calls the model makes before it replies, such as writing to its memory of the user or calling a connector. Each call is followed by a tool result; the counts below are calls only. **The data download leaves these out**, keeping only the visible messages.

In [ ]:
replies = messages[(messages["role"] == "assistant") & ~messages["is_hidden_step"] & messages["resolved_model_slug"].notna()]
(replies.groupby(["default_model_slug", "resolved_model_slug"]).size()
 .rename("replies").reset_index()
 .rename(columns={"default_model_slug": "requested", "resolved_model_slug": "answered by"}))

In [ ]:
reasoning = messages["finished_duration_sec"].dropna()
if reasoning.empty:
    print("No reply in this export reports a reasoning time.")
else:
    fig, ax = plt.subplots(figsize=(6, 2.8))
    ax.hist(reasoning, bins=range(0, int(reasoning.max()) + 2), color=GREEN)
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.set_title(f"Reasoning time ({len(reasoning)} replies report one)")
    ax.set_xlabel("seconds")
    ax.set_ylabel("replies")
    plt.show()

In [ ]:
def step_kind(name):
    if name == "bio":
        return "Memory write"
    if name.startswith("api_tool"):
        return "Connector or tool call"
    if "continue_in_work" in name:
        return "Hand-off to a work session"
    return f"Other tool ({name})"


calls = messages[messages["is_hidden_step"] & (messages["role"] == "assistant")]
kinds = calls["tool_name"].map(step_kind).value_counts()

if kinds.empty:
    print("No hidden steps in this export.")
else:
    fig, ax = plt.subplots(figsize=(7, 0.5 + 0.45 * len(kinds)))
    ax.barh(kinds.index[::-1], kinds.values[::-1], color=BROWN)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.set_title("Hidden steps by kind")
    ax.set_xlabel("calls")
    plt.show()

with_steps = calls["conversation_id"].nunique()
print(f"{with_steps} of {len(conversations)} conversations ({with_steps / max(len(conversations), 1):.0%}) include at least one hidden step.")

## 4. Do people search before they ask?

For each conversation, this looks at browsing in the 30 minutes before and after it started. Sites are grouped rather than named, since a participant's history can include employers, clients, or other sites that identify them.

- **Search**: a Google, Bing or DuckDuckGo visit that came from the address bar or a search box (`generated` or `form_submit`). This is a heuristic; domain-only data shows that a search happened, not what it was for.
- **AI assistant**: ChatGPT, Claude, Gemini, Perplexity or Copilot.
- **Other**: everything else, including category-only visits.

Only the dates covered by both history and conversations are used. When conversations start close together, a visit near several of them is counted once for each.

In [ ]:
SEARCH_DOMAINS = {"google.com", "bing.com", "duckduckgo.com"}
SEARCH_TRANSITIONS = {"generated", "form_submit"}
AI_DOMAINS = {"chatgpt.com", "openai.com", "claude.ai", "gemini.google.com", "perplexity.ai", "copilot.microsoft.com"}
GROUP_COLORS = {"Search": PINK, "AI assistant": TEAL, "Other": MINT}
WINDOW = pd.Timedelta(minutes=30)


def site_group(visit):
    if visit.domain in AI_DOMAINS:
        return "AI assistant"
    if visit.domain in SEARCH_DOMAINS and visit.transition_type in SEARCH_TRANSITIONS:
        return "Search"
    return "Other"


starts = conversations["create_time"]
nearby = visits[(visits["visit_time"] >= starts.min() - WINDOW) & (visits["visit_time"] <= starts.max() + WINDOW)].copy()
nearby["group"] = [site_group(v) for v in nearby.itertuples()]

offsets = []
for start in starts:
    window = nearby[(nearby["visit_time"] - start).abs() <= WINDOW]
    offsets.append(pd.DataFrame({
        "minutes": (window["visit_time"] - start).dt.total_seconds() / 60,
        "group": window["group"],
    }))
offsets = pd.concat(offsets, ignore_index=True) if offsets else pd.DataFrame(columns=["minutes", "group"])

bins = np.arange(-30, 31, 5)
fig, ax = plt.subplots(figsize=(9, 3.6))
bottom = np.zeros(len(bins) - 1)
for group, color in GROUP_COLORS.items():
    counts, _ = np.histogram(offsets.loc[offsets["group"] == group, "minutes"], bins=bins)
    ax.bar(bins[:-1] + 2.5, counts, width=4.6, bottom=bottom, color=color, label=group)
    bottom += counts
ax.axvline(0, color=BROWN, linewidth=1)
ax.text(0.5, ax.get_ylim()[1] * 0.95, "conversation starts", color=BROWN, va="top")
ax.set_xticks(bins)
ax.set_xlabel("minutes from conversation start")
ax.set_ylabel("visits")
ax.set_title("Browsing around each conversation")
ax.legend(frameon=False, ncols=3, loc="upper left", bbox_to_anchor=(0, -0.18))
plt.show()

searched_before = sum(
    ((nearby["group"] == "Search") & (nearby["visit_time"] < start) & (nearby["visit_time"] >= start - pd.Timedelta(minutes=10))).any()
    for start in starts)
print(f"{searched_before} of {len(starts)} conversations ({searched_before / max(len(starts), 1):.0%}) had a search in the 10 minutes before.")

## Reading these results

- **Coverage.** The spider collects conversations back to a configured date, and a crawl can end early if ChatGPT rate-limits it. Check the conversation date range in the first table before comparing it to history.
- **Privacy levels are a design choice.** The shares in section 1 describe the study's lists as much as the participant's behavior.
- **Hidden steps depend on the model's features.** Memory writes only appear when memory is on; connector calls only when connectors are enabled.